# TD4 — Feature engineering EEG pour l'estimation de la charge cognitive

## Contexte

Ce TD s'inscrit dans le projet fil rouge basé sur le papier :

**Multimodal Brain-Computer Interface for In-Vehicle Driver Cognitive Load Measurement: Dataset and Baselines**.

Le papier introduit le dataset **CL-Drive**, dans lequel des signaux EEG, ECG, EDA et Gaze sont enregistrés pendant une tâche de conduite simulée. Les scores de charge cognitive sont collectés toutes les **10 secondes**. Après prétraitement, les signaux sont segmentés en fenêtres de 10 s, puis transformés en caractéristiques numériques pour entraîner des modèles de classification.

Dans ce TD, on se concentre uniquement sur les **features EEG**.

---

## Objectifs pédagogiques

À la fin du TD, vous devez être capables de :

1. expliquer pourquoi on transforme un signal EEG en vecteur de caractéristiques ;
2. distinguer les features temporelles, fréquentielles et non linéaires ;
3. expliquer le principe de la densité spectrale de puissance, ou PSD ;
4. calculer des puissances par bande EEG ;
5. interpréter l'entropie spectrale ;
6. expliquer les paramètres de Hjorth ;
7. comprendre le principe de la complexité de Lempel-Ziv ;
8. comprendre l'idée de la dimension fractale de Higuchi ;
9. structurer une fonction complète d'extraction de features EEG ;
10. préparer une matrice de features pour la classification.

---

## Features EEG ciblées

Le papier regroupe les features EEG suivantes :

| Famille | Features |
|---|---|
| PSD | puissance absolue, moyenne, maximale, minimale et médiane |
| Entropie spectrale | entropie calculée à partir de la PSD normalisée |
| Hjorth | mobility et complexity |
| Lempel-Ziv | complexité d'une séquence binarisée |
| Higuchi | dimension fractale |
| Statistiques temporelles | moyenne, minimum, maximum, médiane, variance, écart-type |

Dans ce TD, on adopte une version complète :

- PSD calculée dans les cinq bandes EEG : delta, theta, alpha, beta, gamma ;
- entropie spectrale calculée dans les cinq bandes ;
- features non linéaires calculées sur le segment temporel ;
- statistiques temporelles calculées sur le segment.

On obtient donc :

$$
5 \text{ bandes} \times 5 \text{ descripteurs PSD} = 25
$$

$$
5 \text{ entropies spectrales} = 5
$$

$$
2 \text{ Hjorth} + 1 \text{ Lempel-Ziv} + 1 \text{ Higuchi} + 6 \text{ statistiques} = 10
$$

Soit au total :

$$
25 + 5 + 10 = 40 \text{ features par canal EEG}
$$

## Questions de compréhension

### Question 1

Pourquoi ne donne-t-on pas directement le signal EEG brut à un classifieur classique comme LDA, SVM ou Random Forest ?

### Réponse 

Un signal EEG de 10 secondes à 256 Hz contient 2560 échantillons par canal, soit 10 240 valeurs pour 4 canaux. Un classifieur comme LDA, SVM ou Random Forest n'est pas conçu pour gérer des entrées de cette dimension brute car il ne sait pas que les échantillons adjacents sont liés dans le temps, il n'exploite pas la structure fréquentielle, et il risque de surapprendre des patterns numériques sans sens physique.

### Question 2

Pourquoi les features fréquentielles sont-elles particulièrement importantes en EEG ?

### Réponse 

Parce que l'activité cérébrale s'organise naturellement par bandes de fréquence qui correspondent à des états cognitifs distincts. La bande alpha (8–12 Hz) est associée à la relaxation, la bande theta (4–8 Hz) à la mémoire de travail, la bande beta (12–30 Hz) à l'attention et l'activité mentale. La charge cognitive modifie ces répartitions spectrales de façon mesurable. Regarder uniquement l'amplitude temporelle ne capture pas ces dynamiques, c'est dans le domaine fréquentiel que le signal EEG est le plus lisible pour ce type de problème.

### Question 3

Pourquoi faut-il calculer les features séparément sur chaque canal EEG ?

### Réponse 

Parce que les 4 canaux du casque Muse (AF7, AF8, TP9, TP10) correspondent à des régions différentes du cerveau. Fusionner les canaux avant l'extraction perdrait l'information spatiale. En calculant 40 features par canal, on obtient 160 features au total qui décrivent à la fois le contenu spectral et la localisation de l'activité.

## 1. Bandes fréquentielles EEG

Les signaux EEG sont souvent analysés par bandes de fréquence.

| Bande | Intervalle utilisé dans ce TD | Interprétation générale |
|---|---:|---|
| Delta | 0.5–4 Hz | activité lente |
| Theta | 4–8 Hz | attention, mémoire de travail, somnolence selon contexte |
| Alpha | 8–12 Hz | relaxation, inhibition, yeux fermés |
| Beta | 12–30 Hz | activité mentale, attention, activité motrice |
| Gamma | 30–75 Hz | activité rapide, intégration, mais sensible aux artefacts musculaires |

Dans le papier, la bande gamma va jusqu'à 75 Hz. 

### Question 4

Pourquoi peut-on limiter la bande gamma à 45 Hz dans certaines implémentations ?

### Réponse 

Au delà de 45 Hz le signal EEG est très facilement contaminé par les artefacts musculaires (EMG).

## 2. Méthodologie d'implémentation 

Dans ce TD, l'objectif est de comprendre puis implémenter les fonctions essentielles.

Pour chaque fonction, vous aurez :

- une explication théorique ;
- l'algorithme ;
- les fonctions Python recommandées ;
- les paramètres importants ;
- des questions de vérification.

Les bibliothèques utiles sont :

| Objectif | Bibliothèque | Fonctions utiles |
|---|---|---|
| Tableaux numériques | NumPy | `np.array`, `np.mean`, `np.var`, `np.diff`, `np.median` |
| Données tabulaires | pandas | `pd.DataFrame`, `pd.read_csv`, `to_csv` |
| PSD | scipy.signal | `welch` |
| Entropie | scipy.stats | `entropy` |
| Visualisation | matplotlib | `plt.plot`, `plt.semilogy`, `plt.bar` |

### Travail à réaliser

Vous devez construire progressivement les fonctions suivantes :

1. `compute_psd_band_features(signal, fs)` ;
2. `compute_spectral_entropy_bands(signal, fs)` ;
3. `compute_hjorth(signal)` ;
4. `lempel_ziv_complexity(signal)` ;
5. `higuchi_fd(signal, kmax=10)` ;
6. `compute_raw_features(signal)` ;
7. `extract_eeg_features(signal, fs)` ;
8. une boucle permettant d'extraire les features sur des fenêtres de 10 secondes.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import entropy

FS = 256
WINDOW_SEC = 10
WINDOW_SAMPLES = FS * WINDOW_SEC

EEG_BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 31),
    "gamma": (31, 75),
}

print("Nombre d'échantillons par fenêtre :", WINDOW_SAMPLES)
print("Bandes EEG :", EEG_BANDS)

Nombre d'échantillons par fenêtre : 2560
Bandes EEG : {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 12), 'beta': (12, 31), 'gamma': (31, 75)}


## 3. Exemples pédagogiques sur signaux connus

Avant d'appliquer les features à l'EEG réel, on commence par des signaux connus.

On va comparer :

1. une sinusoïde à 2 Hz, principalement dans la bande delta ;
2. une sinusoïde à 10 Hz, principalement dans la bande alpha ;
3. une sinusoïde à 20 Hz, principalement dans la bande beta ;
4. un bruit blanc, dont l'énergie est plus répartie ;
5. un mélange de plusieurs composantes.

### Objectif pédagogique

Vérifier que les features donnent des résultats cohérents :

- une sinusoïde pure doit avoir une PSD concentrée ;
- le bruit doit avoir une entropie spectrale plus élevée ;
- une sinusoïde alpha doit avoir une puissance alpha dominante.

In [2]:
# Cellule d'aide : génération de signaux synthétiques pour les tests pédagogiques.
# Vous pouvez l'utiliser pour tester vos futures fonctions de features.

def generate_synthetic_signals(fs=FS, duration=10, random_state=0):
    rng = np.random.default_rng(random_state)
    t = np.arange(0, duration, 1/fs)
    signals = {
        "delta_2Hz": np.sin(2*np.pi*2*t),
        "alpha_10Hz": np.sin(2*np.pi*10*t),
        "beta_20Hz": np.sin(2*np.pi*20*t),
        "white_noise": rng.normal(0, 1, size=len(t)),
        "mixed": 0.8*np.sin(2*np.pi*6*t) + 0.5*np.sin(2*np.pi*10*t) + 0.2*rng.normal(0, 1, size=len(t)),
    }
    return t, signals

t, synthetic_signals = generate_synthetic_signals()

## 4. Feature PSD : densité spectrale de puissance

### Principe théorique

La **PSD**, ou densité spectrale de puissance, indique comment la puissance du signal est répartie selon la fréquence.

Pour un segment EEG $x[n]$, on estime la PSD avec la méthode de Welch. L'idée est de :

1. découper le signal en sous-fenêtres ;
2. calculer un spectre sur chaque sous-fenêtre ;
3. moyenner les spectres pour obtenir une estimation plus stable.

En Python, on utilise généralement :

```python
freqs, psd = scipy.signal.welch(signal, fs=fs, nperseg=...)
```

### Paramètres recommandés

- `fs=256` pour CL-Drive ;
- `nperseg=fs*2` pour une fenêtre Welch de 2 secondes ;
- si le segment est court, prendre `nperseg=min(len(signal), fs*2)` ;
- sélectionner ensuite les fréquences appartenant à une bande donnée à l’aide d’un masque booléen (utiliser le vecteur `freqs` retourné `scipy.signal.welch`).

### Features PSD pour chaque bande

Pour chaque bande EEG, calculer :

1. puissance absolue : somme de la PSD dans la bande ;
2. puissance moyenne ;
3. puissance maximale ;
4. puissance minimale ;
5. puissance médiane.

### Algorithme

1. calculer `freqs, psd` avec Welch ;

Pour chaque bande $[f_{min}, f_{max}]$ :

   
2. sélectionner les indices tels que $f_{min} \le f < f_{max}$ ;
3. extraire `band_psd` ;
4. calculer `sum`, `mean`, `max`, `min`, `median` ;
5. stocker les résultats dans un dictionnaire.

### Questions

1. Quelle bande doit dominer pour un signal sinusoïdal à 10 Hz ?
2. Pourquoi la PSD est-elle plus stable avec Welch qu'avec un simple spectre FFT ?
3. Que se passe-t-il si la bande sélectionnée ne contient aucune fréquence ?

### Réponses 

1. La bande alpha (8–12 Hz). Un sinus à 10 Hz concentre toute son énergie exactement à 10 Hz, qui tombe dans cette bande.
2. Welch découpe le signal en plusieurs petites fenêtres qui se chevauchent, calcule la FFT sur chacune, puis fait la moyenne de tous ces spectres. Cette moyenne réduit le bruit aléatoire. Une FFT directe sur un seul segment donne un spectre très bruité et instable.
3. band_psd est un tableau vide, donc np.sum, np.mean, etc. planteraient ou retourneraient nan. Il faut gérer ce cas en retournant 0 pour tous les descripteurs de cette bande.

In [ ]:
def compute_psd_band_features(signal, fs=256):
    if len(signal) < fs:
        return {}
    freqs, psd = welch(signal, fs=fs, nperseg=min(512, len(signal)), scaling='density')
    features = {}
    for band_name, (fmin, fmax) in EEG_BANDS.items():
        idx = np.logical_and(freqs >= fmin, freqs < fmax)
        band_psd = psd[idx]
        if len(band_psd) == 0:
            features.update({f"{band_name}_{stat}": 0 for stat in ["pow", "mean", "max", "min", "median"]})
            continue
        features.update({
            f"{band_name}_pow":    np.sum(band_psd),
            f"{band_name}_mean":   np.mean(band_psd),
            f"{band_name}_max":    np.max(band_psd),
            f"{band_name}_min":    np.min(band_psd),
            f"{band_name}_median": np.median(band_psd),
        })
    return features

# --- test ---
t, synthetic_signals = generate_synthetic_signals()
psd_feats = compute_psd_band_features(synthetic_signals['alpha_10Hz'])
print('alpha_pow (alpha_10Hz signal):', round(psd_feats['alpha_pow'], 4))
print('beta_pow  (alpha_10Hz signal):', round(psd_feats['beta_pow'],  4))
print('Keys:', list(psd_feats.keys())[:6], '...')

alpha_pow (alpha_10Hz signal): 1.0
beta_pow  (alpha_10Hz signal): 0.0
Keys: ['delta_pow', 'delta_mean', 'delta_max', 'delta_min', 'delta_median', 'theta_pow'] ...


## 5. Entropie spectrale

### Principe théorique

L'entropie spectrale mesure la dispersion de l'énergie dans le domaine fréquentiel.

On calcule d'abord la PSD dans une bande, puis on la normalise pour obtenir une distribution :

$$
p_i = \frac{PSD_i}{\sum_j PSD_j}
$$

L'entropie de Shannon est ensuite :

$$
H = - \sum_i p_i \log(p_i)
$$


### Fonction Python utile

```python
from scipy.stats import entropy
```

`entropy(p)` calcule directement l'entropie de Shannon si `p` est une distribution normalisée.

### Questions

1. Pourquoi faut-il normaliser la PSD avant de calculer l'entropie ?
2. Quel signal devrait avoir l'entropie spectrale la plus élevée : une sinusoïde pure ou un bruit blanc ?
3. Pourquoi l'entropie spectrale peut-elle être utile pour caractériser la complexité d'un EEG ?

### Réponses 

1. La formule de Shannon nécessite une distribution de probabilité : des valeurs entre 0 et 1 qui somment à 1. La PSD brute est une densité d'énergie pas une probabilité. En divisant par la somme totale on la transforme en distribution utilisable.
2. Le bruit blanc. Son énergie est répartie sur toutes les fréquences uniformément donc distribution plate donc entropie maximale. Une sinusoïde concentre toute son énergie sur une seule fréquence => distribution très piquée => entropie proche de 0.
3. Un cerveau en charge cognitive élevée produit des oscillations plus organisées dans certaines bandes (theta notamment) => entropie basse. Au repos, le signal est plus désordonné donc entropie haute. L'entropie permet donc de distinguer des états cognitifs différents.

In [4]:
def compute_spectral_entropy_bands(signal, fs=256):
    freqs, psd = welch(signal, fs=fs, nperseg=min(512, len(signal)))
    features = {}
    for band_name, (fmin, fmax) in EEG_BANDS.items():
        idx = np.logical_and(freqs >= fmin, freqs < fmax)
        band_psd = psd[idx]
        if len(band_psd) == 0 or np.sum(band_psd) == 0:
            features[f"{band_name}_entropy"] = 0
            continue
        p = band_psd / np.sum(band_psd)
        features[f"{band_name}_entropy"] = entropy(p, base=2)
    return features

# --- test ---
ent_feats_sine  = compute_spectral_entropy_bands(synthetic_signals['alpha_10Hz'])
ent_feats_noise = compute_spectral_entropy_bands(synthetic_signals['white_noise'])
print('Entropie alpha (sinus 10Hz)  :', round(ent_feats_sine['alpha_entropy'],  4))
print('Entropie alpha (bruit blanc) :', round(ent_feats_noise['alpha_entropy'], 4))

Entropie alpha (sinus 10Hz)  : 1.2516
Entropie alpha (bruit blanc) : 2.9302


## 6. Paramètres de Hjorth : mobility et complexity

Les paramètres de Hjorth sont des descripteurs temporels utilisés pour caractériser la dynamique d'un signal.

### Mobility

La mobility mesure approximativement la fréquence moyenne du signal :

$$
Mobility(x) = \sqrt{\frac{Var(\Delta x)}{Var(x)}}
$$

où $\Delta x$ est la dérivée discrète du signal, généralement calculée avec `np.diff(x)`.

### Complexity

La complexity mesure la variation de la mobility entre le signal et sa dérivée :

$$
Complexity(x) = \frac{Mobility(\Delta x)}{Mobility(x)}
$$


### Questions

1. Que vaut approximativement la variance d'un signal constant ?
2. Pourquoi faut-il gérer le cas où `Var(x)=0` ?
3. Entre une sinusoïde lisse et un bruit blanc, lequel devrait avoir une complexity plus élevée ?

### Réponses 

1. Tous les échantillons ont la même valeur, donc l'écart à la moyenne est nul partout.
2. Parce que la formule de mobility divise par Var(x). Si c'est 0, on obtient une division par zéro → résultat inf ou nan qui va polluer toute la suite du pipeline.
3. Le bruit blanc. Une sinusoïde est régulière : sa dérivée ressemble au signal lui-même, donc la mobility change peu. Le bruit blanc est irrégulier : ses dérivées successives ont des variances qui explosent → complexity très élevée.

In [5]:
def compute_hjorth(signal):
    signal = np.array(signal)
    if len(signal) < 2 or np.var(signal) == 0:
        return {"hjorth_mobility": 0, "hjorth_complexity": 0}
    diff1 = np.diff(signal)
    mobility = np.sqrt(np.var(diff1) / np.var(signal))
    diff2 = np.diff(diff1)
    complexity = np.sqrt(np.var(diff2) / np.var(diff1)) / mobility if mobility != 0 else 0
    return {"hjorth_mobility": mobility, "hjorth_complexity": complexity}

# --- test ---
h_sine  = compute_hjorth(synthetic_signals['alpha_10Hz'])
h_noise = compute_hjorth(synthetic_signals['white_noise'])
print('Hjorth sinus 10Hz :', h_sine)
print('Hjorth bruit blanc:', h_noise)

Hjorth sinus 10Hz : {'hjorth_mobility': np.float64(0.24477490737748014), 'hjorth_complexity': np.float64(1.0007474243958725)}
Hjorth bruit blanc: {'hjorth_mobility': np.float64(1.4188940906093388), 'hjorth_complexity': np.float64(1.2206898813430527)}


## 7. Complexité de Lempel-Ziv

### Principe théorique

La complexité de Lempel-Ziv mesure le nombre de motifs nouveaux rencontrés dans une séquence.

Comme l'algorithme s'applique à une séquence symbolique, un signal EEG réel doit d'abord être transformé en séquence binaire.

Une méthode simple consiste à binariser le signal par rapport à sa médiane :

$$
b[n] =
\begin{cases}
1, & x[n] > median(x) \\
0, & x[n] \le median(x)
\end{cases}
$$


### Paramètres importants

- seuil de binarisation : médiane ou moyenne ;
- normalisation de la complexité pour comparer des segments de même ou de différente longueur ;
- gestion des segments constants.

### Questions

1. Pourquoi faut-il binariser le signal avant de calculer Lempel-Ziv ?
2. Pourquoi la médiane est-elle un seuil intéressant ?
3. Quel signal devrait avoir une complexité plus élevée : une sinusoïde pure ou un bruit blanc ?
4. Pourquoi normaliser la complexité par la longueur de la séquence ?

### Réponses 

1. L'algorithme de Lempel-Ziv travaille sur des séquences de symboles discrets (0 et 1). Un signal EEG est continu (valeurs réelles) : il faut le transformer en suite binaire avant de pouvoir appliquer l'algorithme.
2. La médiane est robuste aux valeurs extrêmes. Avec la médiane comme seuil, exactement la moitié des échantillons passent à 1 et l'autre moitié à 0, ce qui maximise l'information binaire disponible.
3. Le bruit blanc. Il génère une séquence binaire quasi aléatoire avec plein de nouveaux motifs à chaque pas → complexité élevée. Un sinus donne une séquence très régulière et répétitive => peu de nouveaux motifs => complexité faible.
4. Un signal plus long contient mécaniquement plus de motifs nouveaux, même sans être plus complexe. Diviser par n / log2(n) rend la valeur comparable entre segments de longueurs différentes.

In [6]:
def lempel_ziv_complexity(signal):
    signal = np.array(signal)
    if len(signal) == 0:
        return 0
    median = np.median(signal)
    binary = (signal > median).astype(int)
    s = ''.join(map(str, binary))
    n = len(s)
    if n == 0:
        return 0
    complexity = 1
    s1 = s[0]
    for i in range(1, n):
        s2 = s[:i+1]
        if s2 not in s1:
            complexity += 1
            s1 += s[i]
    return complexity / (n / np.log2(n)) if n > 1 else 0

# --- test ---
lz_sine  = lempel_ziv_complexity(synthetic_signals['alpha_10Hz'])
lz_noise = lempel_ziv_complexity(synthetic_signals['white_noise'])
print('LZ sinus 10Hz :', round(lz_sine,  4))
print('LZ bruit blanc:', round(lz_noise, 4))

LZ sinus 10Hz : 11.3219
LZ bruit blanc: 11.3219


## 8. Dimension fractale de Higuchi

### Principe théorique

La dimension fractale de Higuchi cherche à mesurer la complexité géométrique d'un signal temporel.

Un signal très lisse ressemble davantage à une courbe régulière. Un signal très irrégulier ou bruité présente une trajectoire plus complexe.

L'algorithme de Higuchi construit plusieurs sous-séquences avec différents pas $k$, mesure leur longueur moyenne $L(k)$, puis estime une pente dans un espace logarithmique.

### Paramètre important

- `kmax` : pas maximal testé.

Pour un segment de 10 secondes à 256 Hz, une valeur pédagogique simple est :

```python
kmax = 10
```

Une valeur trop faible peut donner une estimation instable ; une valeur trop élevée augmente le coût de calcul.

### Questions

1. Que cherche à mesurer la dimension fractale de Higuchi ?
2. Pourquoi un signal bruité peut-il avoir une dimension fractale plus élevée qu'une sinusoïde ?
3. Quel est le rôle du paramètre `kmax` ?
4. Pourquoi faut-il éviter de calculer un logarithme de zéro ?

### Réponses 

1. La complexité géométrique du signal dans le temps. Un signal lisse et régulier aura une valeur proche de 1. Un signal très irrégulier et bruité aura une valeur proche de 2 (il "remplit" presque un plan 2D).
2. Le bruit blanc change de direction à chaque échantillon => trajectoire très longue et tortueuse => longueur des sous-séquences varie peu avec le pas k => pente élevée dans l'espace log-log => dimension fractale élevée. Un sinus est lisse => trajectoire simple => dimension proche de 1.
3. Il définit combien de pas différents on teste pour estimer la pente. Trop petit → droite basée sur trop peu de points → estimation instable. Trop grand par rapport à la longueur du signal → sous-séquences trop courtes → longueurs nulles → erreur.
4. log(0) est mathématiquement indéfini (tend vers -∞). Si une sous-séquence a une longueur nulle et qu'on calcule son log, on obtient nan ou -inf qui rend l'ajustement de la droite impossible.

In [7]:
def higuchi_fd(signal, kmax=10):
    signal = np.array(signal)
    n = len(signal)
    if n < kmax * 2:
        return 0
    L = []
    for k in range(1, kmax + 1):
        Lk = []
        for m in range(k):
            Lmk = 0
            for i in range(1, int((n - m) / k)):
                Lmk += abs(signal[m + i*k] - signal[m + (i-1)*k])
            Lmk = Lmk * (n - 1) / (int((n - m)/k) * k)
            Lk.append(Lmk)
        L.append(np.mean(Lk))
    L = np.array(L)
    x = np.log(1.0 / np.arange(1, kmax + 1))
    y = np.log(L)
    if len(x) < 2 or np.any(np.isnan(y)) or np.any(np.isinf(y)):
        return 0
    slope = np.polyfit(x, y, 1)[0]
    return -slope

# --- test ---
hfd_sine  = higuchi_fd(synthetic_signals['alpha_10Hz'])
hfd_noise = higuchi_fd(synthetic_signals['white_noise'])
print('Higuchi FD sinus 10Hz :', round(hfd_sine,  4))
print('Higuchi FD bruit blanc:', round(hfd_noise, 4))

Higuchi FD sinus 10Hz : -0.1073
Higuchi FD bruit blanc: -1.0007


## 9. Statistiques temporelles du signal brut

Les statistiques temporelles simples fournissent des informations directes sur l'amplitude et la variabilité du signal.

Features demandées :

1. moyenne ;
2. minimum ;
3. maximum ;
4. médiane ;
5. variance ;
6. écart-type.

### Fonctions Python utiles

| Feature | Fonction NumPy |
|---|---|
| moyenne | `np.mean` |
| minimum | `np.min` |
| maximum | `np.max` |
| médiane | `np.median` |
| variance | `np.var` |
| écart-type | `np.std` |


In [8]:
def compute_raw_features(signal):
    signal = np.array(signal, dtype=float)
    signal = np.nan_to_num(signal, nan=0.0)
    return {
        "mean":   np.mean(signal),
        "min":    np.min(signal),
        "max":    np.max(signal),
        "median": np.median(signal),
        "var":    np.var(signal),
        "std":    np.std(signal),
    }

# --- test ---
raw = compute_raw_features(synthetic_signals['white_noise'])
print('Raw features:', {k: round(v, 4) for k, v in raw.items()})

Raw features: {'mean': np.float64(-0.0345), 'min': np.float64(-3.8994), 'max': np.float64(3.066), 'median': np.float64(-0.0563), 'var': np.float64(0.9931), 'std': np.float64(0.9965)}


## 10. Fonction complète d'extraction des 40 features EEG

À ce stade, on peut regrouper toutes les familles de features dans une seule fonction.

### Entrée

Un segment EEG 1D correspondant à :

- un canal ;
- une fenêtre de 10 secondes ;
- 2560 échantillons si `fs=256 Hz`.

### Sortie

Un dictionnaire de **40 features**.

### Organisation recommandée

1. convertir le signal en tableau NumPy ;
2. remplacer les valeurs manquantes par 0 ou par une stratégie décidée en amont ;
3. calculer les features PSD par bande ;
4. calculer les entropies spectrales ;
5. calculer Hjorth ;
6. calculer Lempel-Ziv ;
7. calculer Higuchi ;
8. calculer les statistiques temporelles ;
9. fusionner les dictionnaires.

### Questions

1. Pourquoi la fonction doit-elle retourner un dictionnaire plutôt qu'une simple liste ?
2. Pourquoi est-il important de conserver des noms de colonnes explicites ?
3. Combien de features doit retourner la fonction pour un canal ?
4. Si on a 4 canaux et qu'on concatène toutes les features, combien de features obtient-on par segment ?

### Réponses 

1. Un dictionnaire associe chaque valeur à un nom ("alpha_pow", "hjorth_mobility"…). Avec une liste, si l'ordre change ou si on oublie une feature, tout est faussé sans qu'on le voit. Les noms permettent de construire un DataFrame lisible et de retrouver chaque feature facilement.
2. Pour pouvoir interpréter les résultats, calculer les importances de features, déboguer et reproduire l'expérience. Une matrice anonyme de chiffres est une boîte noire impossible à auditer.
3. 40 features : 25 PSD (5 bandes × 5 descripteurs) + 5 entropies spectrales + 2 Hjorth + 1 Lempel-Ziv + 1 Higuchi + 6 statistiques temporelles = 40.
4. 40 × 4 = 160 features par fenêtre de 10 secondes.

In [9]:
def extract_eeg_features(signal, fs=256):
    signal = np.array(signal, dtype=float)
    signal = np.nan_to_num(signal, nan=0.0)
    features = {}
    features.update(compute_psd_band_features(signal, fs))
    features.update(compute_spectral_entropy_bands(signal, fs))
    features.update(compute_hjorth(signal))
    features["lz_complexity"] = lempel_ziv_complexity(signal)
    features["higuchi_fd"]    = higuchi_fd(signal)
    features.update(compute_raw_features(signal))
    return features

# --- vérification : 40 features par canal ? ---
feats = extract_eeg_features(synthetic_signals['alpha_10Hz'])
print(f'Nombre de features: {len(feats)}  (attendu: 40)')
print(list(feats.keys()))

Nombre de features: 40  (attendu: 40)
['delta_pow', 'delta_mean', 'delta_max', 'delta_min', 'delta_median', 'theta_pow', 'theta_mean', 'theta_max', 'theta_min', 'theta_median', 'alpha_pow', 'alpha_mean', 'alpha_max', 'alpha_min', 'alpha_median', 'beta_pow', 'beta_mean', 'beta_max', 'beta_min', 'beta_median', 'gamma_pow', 'gamma_mean', 'gamma_max', 'gamma_min', 'gamma_median', 'delta_entropy', 'theta_entropy', 'alpha_entropy', 'beta_entropy', 'gamma_entropy', 'hjorth_mobility', 'hjorth_complexity', 'lz_complexity', 'higuchi_fd', 'mean', 'min', 'max', 'median', 'var', 'std']


## 11. Comparaison des features sur signaux connus

On applique maintenant la fonction complète aux signaux synthétiques.

### Objectif

Vérifier que :

- `alpha_10Hz` a une puissance alpha élevée ;
- `beta_20Hz` a une puissance beta élevée ;
- `white_noise` a une entropie spectrale élevée ;
- les features non linéaires augmentent généralement avec l'irrégularité.

In [10]:
# Section 11 — Comparaison des features sur les signaux synthétiques
import warnings; warnings.filterwarnings('ignore')

results = {}
for name, sig in synthetic_signals.items():
    results[name] = extract_eeg_features(sig)

df_compare = pd.DataFrame(results).T

# Vérifications clés
print('=== Puissance alpha (doit être max pour alpha_10Hz) ===')
print(df_compare['alpha_pow'].round(4))
print()
print('=== Puissance beta (doit être max pour beta_20Hz) ===')
print(df_compare['beta_pow'].round(4))
print()
print('=== Entropie alpha (doit être élevée pour white_noise) ===')
print(df_compare['alpha_entropy'].round(4))
print()
print('=== Higuchi FD (doit être élevée pour white_noise) ===')
print(df_compare['higuchi_fd'].round(4))

=== Puissance alpha (doit être max pour alpha_10Hz) ===
delta_2Hz      0.000
alpha_10Hz     1.000
beta_20Hz      0.000
white_noise    0.060
mixed          0.253
Name: alpha_pow, dtype: float64

=== Puissance beta (doit être max pour beta_20Hz) ===
delta_2Hz      0.0000
alpha_10Hz     0.0000
beta_20Hz      1.0000
white_noise    0.2949
mixed          0.0123
Name: beta_pow, dtype: float64

=== Entropie alpha (doit être élevée pour white_noise) ===
delta_2Hz      2.8725
alpha_10Hz     1.2516
beta_20Hz      1.5223
white_noise    2.9302
mixed          1.3110
Name: alpha_entropy, dtype: float64

=== Higuchi FD (doit être élevée pour white_noise) ===
delta_2Hz     -0.0071
alpha_10Hz    -0.1073
beta_20Hz     -0.5114
white_noise   -1.0007
mixed         -0.4301
Name: higuchi_fd, dtype: float64


## 12. Application aux signaux EEG du dataset CL-Drive

Après les tests pédagogiques, les mêmes fonctions doivent être appliquées aux signaux EEG prétraités.

### Hypothèse de structure des fichiers

On suppose que les fichiers EEG prétraités sont des fichiers CSV contenant :

- une colonne `Timestamp` ;
- une colonne par canal EEG, par exemple `AF7`, `AF8`, `TP9`, `TP10`.

Exemple de structure :

| Timestamp | AF7 | AF8 | TP9 | TP10 |
|---:|---:|---:|---:|---:|
| 0.000 | ... | ... | ... | ... |
| 0.004 | ... | ... | ... | ... |

### Algorithme d'extraction sur un fichier

1. lire le fichier CSV avec `pd.read_csv` ;
2. identifier les colonnes EEG ;
3. découper le signal en fenêtres de 10 secondes ;
4. pour chaque fenêtre :
   - extraire les 2560 échantillons ;
   - pour chaque canal, calculer les 40 features ;
   - stocker les métadonnées : sujet, fichier, fenêtre, temps début, temps fin, canal ;
5. construire un `DataFrame` ;
6. sauvegarder le résultat en CSV.

### Question

Pourquoi faut-il conserver les colonnes `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` avec les features ?

### Reponse

-Participant : pour faire le LOSO (savoir quel sujet exclure du train).
-File : pour extraire le bon level du scénario et trouver le bon label PAAS.
-Window : pour calculer le timestamp ((Window+1) × 10) et associer le bon score PAAS.
-Channel : pour distinguer les features de chaque canal lors des analyses.
-Start_Time / End_Time : pour un alignement temporel précis si nécessaire.

Sans ces colonnes il est impossible d'associer les features aux labels.

In [11]:
# Section 12 — Fonction de traitement d'un fichier EEG

def process_eeg_file(file_path, participant_id):
    df = pd.read_csv(file_path)
    eeg_channels = [col for col in df.columns if col in ['AF7', 'AF8', 'TP9', 'TP10']]
    if not eeg_channels:
        eeg_channels = [col for col in df.columns if col not in ['Timestamp', 'time', 'index']]

    records = []
    window_size = WINDOW_SAMPLES

    for start in range(0, len(df) - window_size + 1, window_size):
        window = df.iloc[start:start + window_size]
        window_time = start / FS

        for ch in eeg_channels:
            signal = window[ch].values
            features = extract_eeg_features(signal, FS)
            record = {
                "Participant": participant_id,
                "File":        file_path.name,
                "Window":      start // window_size,
                "Start_Time":  window_time,
                "End_Time":    window_time + WINDOW_SEC,
                "Channel":     ch,
            }
            record.update(features)
            records.append(record)

    return pd.DataFrame(records)

print('Fonction process_eeg_file définie avec succès.')
print('Colonnes de sortie: Participant, File, Window, Start_Time, End_Time, Channel + 40 features')

Fonction process_eeg_file définie avec succès.
Colonnes de sortie: Participant, File, Window, Start_Time, End_Time, Channel + 40 features


## 13. Traitement par lot des fichiers EEG prétraités

Dans le projet, les fichiers prétraités peuvent être organisés par sujet.

Exemple :

```text
Data/
|----EEG/
    ├── Participant_ID1/
    │   ├── filtered_scenario_1.csv
    │   ├── filtered_scenario_2.csv
    │   └── ...
    ├── Participant_ID2/
    │   └── ...
|----EDA
|----ECG
|----Gaze
|----Labels
```

### Algorithme par lot

1. parcourir les dossiers sujets ;
2. sélectionner uniquement les fichiers `filtered_*.csv` ;
3. lire chaque fichier ;
4. extraire les features fenêtre par fenêtre ;
6. sauvegarder un fichier CSV de features par sujet.

### Remarque importante

Les labels PAAS ne se trouvent pas dans le même fichier que les signaux. Une étape d’association entre les features et les labels est donc nécessaire dans un second temps : il faut relier les scores de charge cognitive aux intervalles temporels correspondants.

In [12]:
# Section 13 — Traitement par lot (BATCH)
# Lance l'extraction sur tous les fichiers EEG prétraités

import warnings; warnings.filterwarnings('ignore')

EEG_FEATURE_DIR = Path('Data/EEG_Features_10s')
EEG_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print('Démarrage de l\'extraction de features EEG...')

for participant_folder in sorted(Path('Data/EEG').iterdir()):
    if not participant_folder.is_dir():
        continue
    participant_id = participant_folder.name
    print(f'\nParticipant: {participant_id}')

    all_files = list(participant_folder.glob('**/*data_level_*.csv'))
    if not all_files:
        all_files = list(participant_folder.glob('**/*.csv'))

    for data_file in sorted(all_files):
        try:
            print(f'   → {data_file.name}')
            df_features = process_eeg_file(data_file, participant_id)
            if not df_features.empty:
                output_file = EEG_FEATURE_DIR / f'{participant_id}_EEG_features.csv'
                if output_file.exists():
                    df_features.to_csv(output_file, mode='a', header=False, index=False)
                else:
                    df_features.to_csv(output_file, index=False)
                print(f'       {len(df_features)} lignes extraites')
        except Exception as e:
            print(f'       Erreur sur {data_file.name}: {e}')

print('\n TERMINÉ ! Vérifiez le dossier Data/EEG_Features_10s/')

# Résumé des fichiers générés
generated = list(EEG_FEATURE_DIR.glob('*.csv'))
if generated:
    print(f'\nFichiers générés ({len(generated)}):')
    for f in sorted(generated):
        df_check = pd.read_csv(f, nrows=1)
        print(f'  {f.name}  —  {df_check.shape[1]} colonnes')
else:
    print('\nAucun fichier généré — vérifiez que Data/EEG/ contient des fichiers CSV.')

Démarrage de l'extraction de features EEG...

Participant: 1030
   → eeg_data_level_1.csv
       72 lignes extraites
   → eeg_data_level_2.csv
       72 lignes extraites
   → eeg_data_level_3.csv
       72 lignes extraites
   → eeg_data_level_4.csv
       72 lignes extraites
   → eeg_data_level_5.csv
       72 lignes extraites
   → eeg_data_level_6.csv
       72 lignes extraites
   → eeg_data_level_7.csv
       64 lignes extraites
   → eeg_data_level_8.csv
       72 lignes extraites
   → eeg_data_level_9.csv
       72 lignes extraites

Participant: 1105
   → eeg_data_level_1.csv
       72 lignes extraites
   → eeg_data_level_2.csv
       72 lignes extraites
   → eeg_data_level_3.csv
       72 lignes extraites
   → eeg_data_level_4.csv
       64 lignes extraites
   → eeg_data_level_5.csv
       72 lignes extraites
   → eeg_data_level_6.csv
       68 lignes extraites
   → eeg_data_level_7.csv
       72 lignes extraites
   → eeg_data_level_8.csv
       72 lignes extraites
   → eeg_data_le

## 14. Vérifications qualité des features

Avant de passer à la classification, il faut vérifier la qualité de la matrice de features.

### Vérifications recommandées

1. nombre de lignes cohérent avec le nombre de fenêtres et de canaux ;
2. absence de valeurs manquantes ;
3. absence de valeurs infinies ;
4. ordre de grandeur plausible ;
5. nombre de features égal à 40 par canal ;
6. conservation des métadonnées utiles ;
7. possibilité d'associer ensuite chaque fenêtre à un label.

### Questions

1. Pourquoi des valeurs `NaN` peuvent-elles apparaître dans les features ?
2. Pourquoi des valeurs infinies peuvent-elles apparaître ?
3. Que doit-on faire si un segment contient trop de valeurs manquantes ?
4. Pourquoi faut-il éviter de normaliser les features avant la séparation train/test ?

### Reponse


1. Si le signal source contient des données manquantes (déconnexion Bluetooth du casque Muse), ou si une fonction retourne nan suite à une opération invalide (division par zéro non gérée).
2. Principalement dans Hjorth si Var(x) = 0 et limite de division par zéro = inf. Ou lors de la normalisation par la baseline si une moyenne de baseline vaut 0.
3. L'exclure du dataset. Imputer de nombreuses valeurs manquantes fausserait les features (surtout PSD et Higuchi qui dépendent de la forme entière du signal). Le papier CL-Drive exclut directement les segments avec des déconnexions Bluetooth.
4. Si on applique le StandardScaler sur tout le dataset avant la séparation, la moyenne et l'écart-type sont calculés en incluant les données de test. Le modèle accède indirectement à de l'information sur le test set → les performances sont gonflées artificiellement. Le scaler doit être ajusté uniquement sur le train, puis appliqué au test.

In [13]:
# Section 14 — Vérifications qualité

feature_files = list(EEG_FEATURE_DIR.glob('*.csv'))
if not feature_files:
    print('Aucun fichier de features trouvé. Exécutez d\'abord la cellule de batch processing.')
else:
    df_all = pd.concat([pd.read_csv(f) for f in feature_files], ignore_index=True)
    print('=== RÉSUMÉ DE LA MATRICE DE FEATURES ===')
    print(f'Nombre de lignes (fenêtres x canaux) : {len(df_all)}')
    print(f'Nombre de colonnes totales            : {df_all.shape[1]}')
    meta_cols = ['Participant', 'File', 'Window', 'Start_Time', 'End_Time', 'Channel']
    feature_cols = [c for c in df_all.columns if c not in meta_cols]
    print(f'Colonnes de métadonnées               : {len(meta_cols)}')
    print(f'Colonnes de features                  : {len(feature_cols)}  (attendu: 40)')
    print()
    nan_count = df_all[feature_cols].isna().sum().sum()
    inf_count = np.isinf(df_all[feature_cols].select_dtypes(include=[float, int]).values).sum()
    print(f'Valeurs NaN   : {nan_count}')
    print(f'Valeurs inf   : {inf_count}')
    print()
    print('Participants présents:', df_all['Participant'].unique())
    print('Canaux présents      :', df_all['Channel'].unique())
    print()
    print('Aperçu (5 premières lignes) :')
    display(df_all.head())

=== RÉSUMÉ DE LA MATRICE DE FEATURES ===
Nombre de lignes (fenêtres x canaux) : 12556
Nombre de colonnes totales            : 46
Colonnes de métadonnées               : 6
Colonnes de features                  : 40  (attendu: 40)

Valeurs NaN   : 0
Valeurs inf   : 0

Participants présents: [1030 1105 1106 1241 1271 1314 1323 1337 1372 1417 1434 1544 1547 1595
 1629 1716 1717 1744 1868 1892 1953]
Canaux présents      : ['TP9' 'AF7' 'AF8' 'TP10']

Aperçu (5 premières lignes) :


,Participant,File,Window,Start_Time,End_Time,Channel,delta_pow,delta_mean,delta_max,delta_min,...,hjorth_mobility,hjorth_complexity,lz_complexity,higuchi_fd,mean,min,max,median,var,std
0,1030,eeg_data_level_1.csv,0,0.0,10.0,TP9,2726.690058,389.527151,1224.651028,150.053743,...,0.285397,4.736889,11.321928,-0.697670,-19.917488,-269.042969,190.917969,-26.367188,3240.390047,56.924424
1,1030,eeg_data_level_1.csv,0,0.0,10.0,AF7,305.508778,43.644111,92.959505,13.898414,...,0.502841,2.535406,11.321928,-0.730531,-20.692444,-95.703125,52.246094,-20.507812,303.682496,17.426488
2,1030,eeg_data_level_1.csv,0,0.0,10.0,AF8,585.943415,83.706202,105.701601,52.037081,...,0.491598,2.631816,11.321928,-0.741346,-17.578888,-105.468750,120.605469,-17.578125,598.824955,24.470900
3,1030,eeg_data_level_1.csv,0,0.0,10.0,TP10,1459.946168,208.563738,318.585456,125.646682,...,0.483457,2.913834,11.321928,-0.761774,-20.135880,-258.300781,117.675781,-18.554688,1418.860131,37.667760
4,1030,eeg_data_level_1.csv,1,10.0,20.0,TP9,3166.055472,452.293639,554.449619,374.010425,...,0.340017,3.868602,11.321928,-0.617379,-23.740005,-329.589844,136.718750,-20.019531,2489.655784,49.896451
